# La API en un entorno empresarial: un nivel por decisión de negocio

`api/main.py` sirve un modelo LightGBM independiente por nivel de la jerarquía M5
(ver `config/levels.py`): desde ventas totales de la cadena (nivel 1) hasta
tienda×categoría (nivel 8 -- los niveles ya entrenados hoy en `artifacts/models/`,
ver `notebooks/04_hierarchical_comparison.ipynb` para su comparación de error).
Cada nivel resuelve una decisión de negocio distinta, con un dueño y una
frecuencia distintos: no tiene sentido pedirle al gerente de una tienda que
planifique con la serie total de la cadena, ni a Finanzas que arme el guidance
trimestral sumando a mano 10 tiendas × 3 categorías.

Este notebook recorre un escenario de negocio concreto por nivel, llamando a la
API real levantada en este mismo proceso (no al modelo directamente) tal como lo
haría un sistema externo (ERP, planillas de compras, dashboard de turnos).

**Alcance**: la API espera el vector de features ya calculado -- no reconstruye
lags/rolling/target-encoding desde datos crudos (ver `docs/informe.md` §9). Para
armar un payload realista, cada escenario toma una fila real del *test* de
backtest de ese nivel (mismo enfoque que `scripts/test_api.py`) y la manda tal
cual a `/predict/{level_id}`; en producción ese vector lo arma el pipeline de
feature engineering, no este notebook. Como la fila es de test, además de la
predicción tenemos el real observado ese día -- lo mostramos para que cada
escenario se lea con el mismo espíritu crítico que el resto del TFM, no como una
demo de caja negra.

In [ ]:
import threading
import time

import joblib
import pandas as pd
import requests
import uvicorn
from loguru import logger

import config
from api.main import app
from src.data.split import reconstruct_test_data

pd.set_option("display.max_columns", None)

## 0. Levantar la API

Mismo `app` de `api/main.py` (mismo código que corre `make serve-api`), pero en
un thread de este proceso para que el notebook sea autocontenido -- no depende de
tener `make serve-api` corriendo en otra terminal.

In [ ]:
BASE_URL = "http://127.0.0.1:8010"

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8010, log_level="warning"))
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

for _ in range(50):
    try:
        requests.get(f"{BASE_URL}/health", timeout=0.5)
        break
    except requests.exceptions.ConnectionError:
        time.sleep(0.1)
else:
    raise RuntimeError("La API no respondió -- ¿hay otro proceso en el puerto 8010?")

logger.success("API arriba en {}", BASE_URL)

2026-09-05 22:45:11.354 | INFO     | api.main:lifespan:27 - API iniciada


2026-09-05 22:45:11.424 | INFO     | api.main:log_requests:39 - GET /health -> 200 (6.3 ms)


2026-09-05 22:45:11.434 | SUCCESS  | __main__:<module>:16 - API arriba en http://127.0.0.1:8010


## 1. Qué nivel usar para qué

`GET /levels` devuelve los niveles con artifact entrenado (los que faltan --
9 a 12, ver `config/levels.py`, densidad item-level -- se filtran solos: el
endpoint hace `try/except FileNotFoundError` por nivel). Le sumamos, a mano, para
qué área de negocio serviría cada uno en una operación tipo retail.

In [ ]:
NEGOCIO = {
    1: ("Dirección financiera", "Guidance de ventas consolidado para el cierre mensual."),
    2: ("Operaciones regionales", "Asignación de flota/inventario entre estados (CA/TX/WI)."),
    3: ("Category management", "Plan de compras por categoría (FOODS/HOBBIES/HOUSEHOLD)."),
    4: ("Trade marketing", "Metros de góndola por departamento dentro de una categoría."),
    5: ("Category management regional", "Plan de compras por categoría, abierto por estado."),
    6: ("Gerencia de tienda", "Dotación de personal y turnos por tienda."),
    7: ("Planificación regional", "Presupuesto por departamento, abierto por estado."),
    8: ("Reposición", "Cuánto pedir de una categoría en una tienda concreta."),
}

levels_resp = requests.get(f"{BASE_URL}/levels").json()
df_levels = pd.DataFrame(levels_resp).sort_values("level_id")
df_levels["area_de_negocio"] = df_levels["level_id"].map(lambda i: NEGOCIO.get(i, ("-", "-"))[0])
df_levels["decision_tipica"] = df_levels["level_id"].map(lambda i: NEGOCIO.get(i, ("-", "-"))[1])
df_levels[["level_id", "level", "area_de_negocio", "decision_tipica", "wape_test", "wrmsse_test"]]

2026-09-05 22:45:13.430 | INFO     | api.main:log_requests:39 - GET /levels -> 200 (1971.5 ms)


,level_id,level,area_de_negocio,decision_tipica,wape_test,wrmsse_test
0,1,level_01_daily_total,Dirección financiera,Guidance de ventas consolidado para el cierre ...,0.059545,0.579390
1,2,level_02_daily_state,Operaciones regionales,Asignación de flota/inventario entre estados (...,0.058795,0.549077
2,3,level_03_daily_cat,Category management,Plan de compras por categoría (FOODS/HOBBIES/H...,0.056743,0.493992
3,4,level_04_daily_dept,Trade marketing,Metros de góndola por departamento dentro de u...,0.065183,0.596292
4,5,level_05_daily_state_cat,Category management regional,"Plan de compras por categoría, abierto por est...",0.069812,0.589583
5,6,level_06_daily_store,Gerencia de tienda,Dotación de personal y turnos por tienda.,0.073069,0.613509
6,7,level_07_daily_state_dept,Planificación regional,"Presupuesto por departamento, abierto por estado.",0.076791,0.598247
7,8,level_08_daily_store_cat,Reposición,Cuánto pedir de una categoría en una tienda co...,0.077331,0.577627


## 2. Helpers de consumo

`predict_series` reproduce, por cada serie pedida, el mismo camino que un cliente
externo: arma el payload de features desde la fila real de test (tal como
`scripts/test_api.py`), llama a `POST /predict/{level_id}` por HTTP, y devuelve
predicción vs. real en la última fecha del backtest de ese nivel.

Ojo con el `error_pct` de cada escenario: es un único día (2016-05-22), no el
`wape_test` agregado de 28 días de la sección 1 -- la varianza día a día es
normal y no implica que el modelo haya empeorado.

In [ ]:
def load_level_test(level_id: int, target: str = "sales"):
    path = sorted(config.MODELS_DIR.glob(f"level_{level_id:02d}_*_{target}_artifact.pkl"))[0]
    artifact = joblib.load(path)
    X_test, y_test, train, test = reconstruct_test_data(artifact)
    return artifact, X_test, test


def to_feature_value(value, is_categorical: bool):
    # None (json null) en vez de NaN: requests.post(json=...) no serializa NaN (a
    # diferencia de json.dumps estándar) -- ver api/main.py para cómo reconstruye
    # una categórica faltante sin romper la inferencia de XGBoost.
    if pd.isna(value):
        return None
    return str(value) if is_categorical else float(value)


def predict_via_api(level_id: int, artifact: dict, x_row: pd.Series) -> float:
    features = {
        col: to_feature_value(x_row[col], col in artifact["categorical_features"])
        for col in artifact["features"]
    }
    resp = requests.post(f"{BASE_URL}/predict/{level_id}", json={"features": features})
    resp.raise_for_status()
    return resp.json()["prediction"]


def predict_series(level_id: int, series_ids: list[str]) -> pd.DataFrame:
    artifact, X_test, test = load_level_test(level_id)
    last_date = test["date"].max()
    rows = []
    for sid in series_ids:
        idx = test.index[(test["series_id"] == sid) & (test["date"] == last_date)][0]
        pred = predict_via_api(level_id, artifact, X_test.loc[idx])
        actual = float(test.loc[idx, "sales"])
        rows.append({
            "series_id": sid, "date": last_date.date(),
            "prediccion": round(pred, 1), "real": actual,
            "error_pct": round(100 * (pred - actual) / actual, 1),
        })
    return pd.DataFrame(rows)

## 3. Escenario -- Nivel 1 (total): guidance financiero

Finanzas necesita, antes de cerrar el mes, una proyección de ventas de la
cadena completa sin esperar el consolidado real del día. Una sola llamada a
`/predict/1` reemplaza sumar a mano 10 tiendas × 3 categorías.

In [ ]:
df_total = predict_series(1, ["TOTAL"])
df_total

2026-09-05 22:45:13.771 | INFO     | src.data.split:load_data:359 - level_01_daily_total | target=sales | 1,941 filas | 191 features (9 categóricas)


2026-09-05 22:45:13.788 | INFO     | src.data.split:log_summary:48 - Train : 2011-01-29 to 2015-04-26 (1,549 days, 1,549 rows)


2026-09-05 22:45:13.790 | INFO     | src.data.split:log_summary:49 - Valid : 2015-04-27 to 2016-04-24 (364 days, 364 rows)


2026-09-05 22:45:13.794 | INFO     | src.data.split:log_summary:50 - Test  : 2016-04-25 to 2016-05-22 (28 days, 28 rows)


2026-09-05 22:45:14.212 | INFO     | src.data.split:split_data:428 - Features: 182 (9 categóricas)


2026-09-05 22:45:14.349 | INFO     | api.main:log_requests:39 - POST /predict/1 -> 200 (125.3 ms)


,series_id,date,prediccion,real,error_pct
0,TOTAL,2016-05-22,44694.2,54338.0,-17.7


## 4. Escenario -- Nivel 2 (estado): asignación de flota regional

Operaciones reparte camiones/inventario entre CA, TX y WI para la semana. La
predicción por estado se traduce directo en `share_pct`: la proporción de flota
que le toca a cada uno.

In [ ]:
df_estado = predict_series(2, ["CA", "TX", "WI"])
df_estado["share_flota_pct"] = (100 * df_estado["prediccion"] / df_estado["prediccion"].sum()).round(1)
df_estado

2026-09-05 22:45:14.579 | INFO     | src.data.split:load_data:359 - level_02_daily_state | target=sales | 5,823 filas | 191 features (9 categóricas)


2026-09-05 22:45:14.597 | INFO     | src.data.split:log_summary:48 - Train : 2011-01-29 to 2015-04-26 (1,549 days, 4,647 rows)


2026-09-05 22:45:14.599 | INFO     | src.data.split:log_summary:49 - Valid : 2015-04-27 to 2016-04-24 (364 days, 1,092 rows)


2026-09-05 22:45:14.603 | INFO     | src.data.split:log_summary:50 - Test  : 2016-04-25 to 2016-05-22 (28 days, 84 rows)


2026-09-05 22:45:14.993 | INFO     | src.data.split:split_data:428 - Features: 183 (9 categóricas)


2026-09-05 22:45:15.151 | INFO     | api.main:log_requests:39 - POST /predict/2 -> 200 (141.6 ms)


2026-09-05 22:45:15.282 | INFO     | api.main:log_requests:39 - POST /predict/2 -> 200 (118.1 ms)


2026-09-05 22:45:15.416 | INFO     | api.main:log_requests:39 - POST /predict/2 -> 200 (122.0 ms)


,series_id,date,prediccion,real,error_pct,share_flota_pct
0,CA,2016-05-22,20410.4,24644.0,-17.2,43.0
1,TX,2016-05-22,12851.2,14815.0,-13.3,27.1
2,WI,2016-05-22,14173.8,14879.0,-4.7,29.9


## 5. Escenario -- Nivel 3 (categoría): plan de compras

Category management arma la orden de compra semanal por categoría con un
colchón de seguridad (`SAFETY_STOCK`) sobre la demanda pronosticada, para no
quedarse corto ante un pico de ventas no capturado por el forecast puntual.

In [ ]:
SAFETY_STOCK = 1.15

df_categoria = predict_series(3, ["FOODS", "HOBBIES", "HOUSEHOLD"])
df_categoria["orden_sugerida"] = (df_categoria["prediccion"] * SAFETY_STOCK).round(0)
df_categoria

2026-09-05 22:45:15.621 | INFO     | src.data.split:load_data:359 - level_03_daily_cat | target=sales | 5,823 filas | 191 features (9 categóricas)


2026-09-05 22:45:15.636 | INFO     | src.data.split:log_summary:48 - Train : 2011-01-29 to 2015-04-26 (1,549 days, 4,647 rows)


2026-09-05 22:45:15.639 | INFO     | src.data.split:log_summary:49 - Valid : 2015-04-27 to 2016-04-24 (364 days, 1,092 rows)


2026-09-05 22:45:15.642 | INFO     | src.data.split:log_summary:50 - Test  : 2016-04-25 to 2016-05-22 (28 days, 84 rows)


2026-09-05 22:45:16.056 | INFO     | src.data.split:split_data:428 - Features: 183 (9 categóricas)


2026-09-05 22:45:16.206 | INFO     | api.main:log_requests:39 - POST /predict/3 -> 200 (135.8 ms)


2026-09-05 22:45:16.359 | INFO     | api.main:log_requests:39 - POST /predict/3 -> 200 (138.5 ms)


2026-09-05 22:45:16.499 | INFO     | api.main:log_requests:39 - POST /predict/3 -> 200 (127.3 ms)


,series_id,date,prediccion,real,error_pct,orden_sugerida
0,FOODS,2016-05-22,32235.2,35967.0,-10.4,37070.0
1,HOBBIES,2016-05-22,4772.8,5280.0,-9.6,5489.0
2,HOUSEHOLD,2016-05-22,12713.8,13091.0,-2.9,14621.0


## 6. Escenario -- Nivel 4 (departamento): espacio de góndola

Trade marketing decide cuánto espacio de góndola darle a cada departamento
dentro de HOUSEHOLD, proporcional a la demanda esperada de cada uno.

In [ ]:
df_depto = predict_series(4, ["HOUSEHOLD_1_HOUSEHOLD", "HOUSEHOLD_2_HOUSEHOLD"])
df_depto["espacio_gondola_pct"] = (100 * df_depto["prediccion"] / df_depto["prediccion"].sum()).round(1)
df_depto

2026-09-05 22:45:16.657 | INFO     | src.data.split:load_data:359 - level_04_daily_dept | target=sales | 13,587 filas | 191 features (9 categóricas)


2026-09-05 22:45:16.675 | INFO     | src.data.split:log_summary:48 - Train : 2011-01-29 to 2015-04-26 (1,549 days, 10,843 rows)


2026-09-05 22:45:16.678 | INFO     | src.data.split:log_summary:49 - Valid : 2015-04-27 to 2016-04-24 (364 days, 2,548 rows)


2026-09-05 22:45:16.681 | INFO     | src.data.split:log_summary:50 - Test  : 2016-04-25 to 2016-05-22 (28 days, 196 rows)


2026-09-05 22:45:17.097 | INFO     | src.data.split:split_data:428 - Features: 165 (9 categóricas)


2026-09-05 22:45:17.229 | INFO     | api.main:log_requests:39 - POST /predict/4 -> 200 (117.0 ms)


2026-09-05 22:45:17.365 | INFO     | api.main:log_requests:39 - POST /predict/4 -> 200 (120.9 ms)


,series_id,date,prediccion,real,error_pct,espacio_gondola_pct
0,HOUSEHOLD_1_HOUSEHOLD,2016-05-22,10016.9,10165.0,-1.5,79.8
1,HOUSEHOLD_2_HOUSEHOLD,2016-05-22,2537.4,2926.0,-13.3,20.2


## 7. Escenario -- Nivel 6 (tienda): dotación de personal

El gerente de zona decide, tienda por tienda, si conviene sumar un turno extra
de caja. Regla simple: por encima de `UMBRAL_TURNO_EXTRA` unidades/día, refuerzo
de personal.

In [ ]:
UMBRAL_TURNO_EXTRA = 6_000

df_tienda = predict_series(6, ["CA_1_CA", "CA_2_CA", "CA_3_CA", "CA_4_CA"])
df_tienda["turno_extra"] = df_tienda["prediccion"] > UMBRAL_TURNO_EXTRA
df_tienda

2026-09-05 22:45:17.549 | INFO     | src.data.split:load_data:359 - level_06_daily_store | target=sales | 19,410 filas | 191 features (9 categóricas)


2026-09-05 22:45:17.577 | INFO     | src.data.split:log_summary:48 - Train : 2011-01-29 to 2015-04-26 (1,549 days, 15,490 rows)


2026-09-05 22:45:17.579 | INFO     | src.data.split:log_summary:49 - Valid : 2015-04-27 to 2016-04-24 (364 days, 3,640 rows)


2026-09-05 22:45:17.582 | INFO     | src.data.split:log_summary:50 - Test  : 2016-04-25 to 2016-05-22 (28 days, 280 rows)


2026-09-05 22:45:18.038 | INFO     | src.data.split:split_data:428 - Features: 191 (9 categóricas)


2026-09-05 22:45:18.197 | INFO     | api.main:log_requests:39 - POST /predict/6 -> 200 (135.2 ms)


2026-09-05 22:45:18.345 | INFO     | api.main:log_requests:39 - POST /predict/6 -> 200 (136.2 ms)


2026-09-05 22:45:18.496 | INFO     | api.main:log_requests:39 - POST /predict/6 -> 200 (139.0 ms)


2026-09-05 22:45:18.658 | INFO     | api.main:log_requests:39 - POST /predict/6 -> 200 (148.9 ms)


,series_id,date,prediccion,real,error_pct,turno_extra
0,CA_1_CA,2016-05-22,5609.4,6289.0,-10.8,False
1,CA_2_CA,2016-05-22,5558.8,6614.0,-16.0,False
2,CA_3_CA,2016-05-22,6228.9,8144.0,-23.5,True
3,CA_4_CA,2016-05-22,3185.8,3597.0,-11.4,False


## 8. Escenario -- Nivel 8 (tienda×categoría): reposición fina

El sistema de reposición de una tienda concreta pide, para una categoría
concreta, la cantidad a reponer en las próximas 48 horas (`LEAD_TIME_DIAS`) en
base al pronóstico diario -- la granularidad más fina con artifact entrenado hoy.

In [ ]:
LEAD_TIME_DIAS = 2

df_reposicion = predict_series(8, ["FOODS_CA_1_CA"])
df_reposicion["pedido_48h_sugerido"] = (df_reposicion["prediccion"] * LEAD_TIME_DIAS).round(0)
df_reposicion

2026-09-05 22:45:18.983 | INFO     | src.data.split:load_data:359 - level_08_daily_store_cat | target=sales | 58,230 filas | 191 features (9 categóricas)


2026-09-05 22:45:19.105 | INFO     | src.data.split:log_summary:48 - Train : 2011-01-29 to 2015-04-26 (1,549 days, 46,470 rows)


2026-09-05 22:45:19.109 | INFO     | src.data.split:log_summary:49 - Valid : 2015-04-27 to 2016-04-24 (364 days, 10,920 rows)


2026-09-05 22:45:19.114 | INFO     | src.data.split:log_summary:50 - Test  : 2016-04-25 to 2016-05-22 (28 days, 840 rows)


2026-09-05 22:45:19.917 | INFO     | src.data.split:split_data:428 - Features: 128 (7 categóricas)


2026-09-05 22:45:20.030 | INFO     | api.main:log_requests:39 - POST /predict/8 -> 200 (96.0 ms)


,series_id,date,prediccion,real,error_pct,pedido_48h_sugerido
0,FOODS_CA_1_CA,2016-05-22,3856.8,4327.0,-10.9,7714.0


## 9. Resumen: de decisión de negocio a nivel de la API

| Decisión de negocio | Nivel | Endpoint | Quién lo consume |
|---|---|---|---|
| Guidance financiero consolidado | 1 (total) | `/predict/1` | Dirección financiera |
| Asignación de flota entre estados | 2 (state) | `/predict/2` | Operaciones regionales |
| Plan de compras por categoría | 3 (cat) | `/predict/3` | Category management |
| Espacio de góndola por departamento | 4 (dept) | `/predict/4` | Trade marketing |
| Dotación de personal por tienda | 6 (store) | `/predict/6` | Gerencia de tienda |
| Reposición tienda×categoría | 8 (store_cat) | `/predict/8` | Sistema de reposición |

La regla general: **cuanto más operativa y de corto plazo es la decisión, más
fino el nivel** -- y más alto el error esperado (`wape_test` sube de nivel 1 a
nivel 8 en la tabla de la sección 1, ver también
`notebooks/04_hierarchical_comparison.ipynb`), porque cada serie individual
tiene menos volumen para promediar el ruido. La API no fuerza a elegir un único
nivel: distintas áreas de la misma empresa consultan, el mismo día, endpoints
distintos según la granularidad de su propia decisión.

## 10. Apagado

El servidor corre en un thread daemon de este proceso -- muere solo al cerrar
el kernel, pero lo apagamos explícitamente para liberar el puerto si se vuelve a
correr esta celda.

In [ ]:
server.should_exit = True
thread.join(timeout=5)
logger.info("API detenida")

2026-09-05 22:45:20.216 | INFO     | __main__:<module>:3 - API detenida
